# Collections Recovery Forensic Analysis

This notebook rebuilds the main findings from the supplied collections data. The analysis separates data-quality corrections from business performance and treats the targeting result as observational unless a randomized holdout is used.

Evidence labels used below: Fact, Strong Evidence, Correlation, Hypothesis.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'raw').exists():
    REPO_ROOT = REPO_ROOT.parent
RAW = Path(os.getenv('CREDRESOLVE_RAW_DIR', str(REPO_ROOT / 'raw'))).resolve()
DATA = REPO_ROOT / 'golden_dataset'
RESULTS = REPO_ROOT / 'reports' / 'results'
OUT = RESULTS
files = ['accounts','borrowers','agents','agent_sessions','campaigns','daily_targeting','calls','call_attempts','call_dispositions','whatsapp_events','sms_events','field_visits','promises_to_pay','payments','vendor_telephony','complaints','account_status_history']
d = {f: pd.read_csv(RAW / f'{f}.csv') for f in files}
accounts = d['accounts']
targeting = d['daily_targeting'].copy()
targeting['target_date'] = pd.to_datetime(targeting['target_date'])
payments = d['payments']
golden = pd.read_csv(DATA / 'golden_payments.csv', parse_dates=['event_at'])
print('Repository layout resolved: raw/, golden_dataset/, reports/results/')


Repository layout resolved: raw/, golden_dataset/, reports/results/


## 1. Raw data checks
The source package is intentionally messy. First quantify the defects before using the data for business conclusions.

In [2]:
p=d['payments']; c=d['calls']; wa=d['whatsapp_events']; st=d['account_status_history']; ag=d['agents']; ac=d['accounts']; br=d['borrowers']
print('duplicate payment_id rows:', len(p)-p.payment_id.nunique())
print('duplicate call_id rows:', len(c)-c.call_id.nunique())
print('duplicate whatsapp_event_id rows:', len(wa)-wa.whatsapp_event_id.nunique())
print('status recorded_at < event_at:', (pd.to_datetime(st.recorded_at)<pd.to_datetime(st.event_at)).sum())
print('orphan accounts:', ac[~ac.borrower_id.isin(br.borrower_id)].shape[0])
print('borrower rows / unique ids:', len(br), br.borrower_id.nunique())
print('agent rows / unique ids:', len(ag), ag.agent_id.nunique())

duplicate payment_id rows: 500
duplicate call_id rows: 1350
duplicate whatsapp_event_id rows: 600
status recorded_at < event_at: 30191
orphan accounts: 2913
borrower rows / unique ids: 30600 11015
agent rows / unique ids: 30000 1000


## 2. Audited payment layer
Exact duplicate payment IDs are removed. Reused payment references that span distinct accounts or distinct amounts are treated as ambiguous and excluded from the audited SUCCESS cash total. This is deliberately conservative.

In [3]:
p2=p.sort_values(['payment_id','event_at']).drop_duplicates('payment_id', keep='first').copy()
refs=(p2[p2.payment_reference.fillna('').str.strip()!='']
      .groupby('payment_reference').agg(accounts=('account_id','nunique'), amounts=('amount','nunique')).reset_index())
bad_refs=set(refs.loc[(refs.accounts>1)|(refs.amounts>1),'payment_reference'])
p2['audited_recovery']=((p2.payment_status.eq('SUCCESS')) & (~p2.payment_reference.fillna('').isin(bad_refs) | p2.payment_reference.fillna('').eq(''))).astype(int)
raw_success=p.loc[p.payment_status.eq('SUCCESS'),'amount'].sum()
audited_success=p2.loc[p2.audited_recovery.eq(1),'amount'].sum()
print('raw SUCCESS cash:', raw_success)
print('audited SUCCESS cash:', audited_success)
print('raw to audited %:', 100*(audited_success/raw_success-1))
print('ambiguous references:', len(bad_refs))

raw SUCCESS cash: 1341485926.33
audited SUCCESS cash: 935251526.3299999
raw to audited %: -30.282419817207085
ambiguous references: 3406


## 3. Independent monthly recovery definition
Primary metric: audited SUCCESS cash by event month. Secondary metric: audited cash on targeted accounts divided by targeted portfolio principal. Both are reported so the business claim can be tested without assuming its denominator.

In [4]:
m=pd.read_csv(OUT/'monthly_metrics.csv')
print(m[['month','audited_recovery_cash_all_accounts','audited_recovery_cash_on_targeted_accounts','targeted_accounts','target_account_recovery_yield']].to_string(index=False))
j=m[m.month.isin(['2026-06','2026-07'])].set_index('month')
print('Jun-Jul audited cash MoM %:', 100*(j.loc['2026-07','audited_recovery_cash_all_accounts']/j.loc['2026-06','audited_recovery_cash_all_accounts']-1))
print('Jun-Jul targeted yield relative %:', 100*(j.loc['2026-07','target_account_recovery_yield']/j.loc['2026-06','target_account_recovery_yield']-1))
print('Jan-Jul audited cash change %:', 100*(m.loc[m.month=='2026-07','audited_recovery_cash_all_accounts'].iat[0]/m.loc[m.month=='2026-01','audited_recovery_cash_all_accounts'].iat[0]-1))

  month  audited_recovery_cash_all_accounts  audited_recovery_cash_on_targeted_accounts  targeted_accounts  target_account_recovery_yield
2026-01                        133703351.45                                 24757294.05               5732                       0.010715
2026-02                        121151851.27                                 20254790.35               5160                       0.009681
2026-03                        135611823.95                                 27908565.36               5666                       0.012114
2026-04                        123812563.84                                 23083482.43               5585                       0.010418
2026-05                        130570578.39                                 24238009.86               5800                       0.010365
2026-06                        125639874.64                                 23805483.54               5535                       0.010660
2026-07                        130

## 4. Mix check
The supplied decomposition is retained as a transparent DPD mix check. The goal is to see whether the Jan to Jul targeted-yield change can be explained by DPD composition.

In [5]:
mix=pd.read_csv(OUT/'mix_decomposition.csv')
print(mix.to_string(index=False))

              metric    value
  jan_observed_yield 0.010715
  jul_observed_yield 0.011918
jul_yield_at_jan_mix 0.011917
jan_yield_at_jul_mix 0.010739
  observed_change_pp 0.120284
within_mix_change_pp 0.120230
       mix_effect_pp 0.000054


## 5. Channel, priority and calling-time checks

In [6]:
print('Channel 7-day conversion:')
print(pd.read_csv(OUT/'channel_7d_conversion.csv').to_string(index=False))
print()
print('July priority bands:')
print(pd.read_csv(OUT/'priority_monthly.csv').query("m=='2026-07'").to_string(index=False))
print()
print('Calling hours with highest answer rate:')
print(pd.read_csv(OUT/'calling_hour.csv').sort_values('answer_rate',ascending=False).head(10).to_string(index=False))

Channel 7-day conversion:
 channel  unique_touches  touches_with_7d_audited_payment  7d_touch_conversion  cash_within_7d
   VOICE           90011                             1146             0.012732     87892217.07
WHATSAPP           60000                              752             0.012533     56322749.63
     SMS           45000                              582             0.012933     43637941.45
   FIELD           25000                              321             0.012840     24143929.95

July priority bands:
      m priority_band  targeted_accounts  paying_accounts        cash    principal    yield
2026-07           1-4               2240              140 11165545.34 888035179.36 0.012573
2026-07           5-7               1800              107  8719023.38 715860728.08 0.012180
2026-07          8-10               1626              101  7212961.93 669769547.85 0.010769

Calling hours with highest answer rate:
      m  local_hour  calls  answer_rate
2026-08          21    120  

## 6. Time-aligned targeting analysis

The legacy July payer comparison is **not causal** because the full-month July outcome can occur before the first July targeting event. That metric is retained only as a diagnostic in `legacy_propensity_sensitivity.csv`.

The corrected historical signal indexes each targeted account at its **first July target date** and measures seven full calendar days before and seven full calendar days after that date, excluding the entire target day because `target_date` has no timestamp. This is still observational; it can reflect seasonality, regression to the mean, selection, or other confounding.


In [7]:
july = targeting[targeting.target_date.dt.to_period('M').astype(str).eq('2026-07')]
first_target = july.groupby('account_id', as_index=False).target_date.min().rename(columns={'target_date':'first_target_date'})
recovery = golden[(golden.payment_status.eq('SUCCESS')) & (golden.audited_recovery.eq(1))][['account_id','event_at','amount']]
x = first_target.merge(recovery, on='account_id', how='left')
x['pre7'] = x.event_at.ge(x.first_target_date - pd.Timedelta(days=7)) & x.event_at.lt(x.first_target_date)
x['post7'] = x.event_at.gt(x.first_target_date) & x.event_at.le(x.first_target_date + pd.Timedelta(days=7))
prepost = x.groupby('account_id').agg(
    pre7_payer=('pre7','max'), post7_payer=('post7','max'),
    pre7_cash=('amount', lambda s: s[x.loc[s.index,'pre7']].sum()),
    post7_cash=('amount', lambda s: s[x.loc[s.index,'post7']].sum())
).reset_index()
print('Pre-7d payer rate:', prepost.pre7_payer.mean())
print('Post-7d payer rate:', prepost.post7_payer.mean())
print('Payer lift (pp):', 100*(prepost.post7_payer.mean()-prepost.pre7_payer.mean()))
print('Pre/post average cash per targeted account:', prepost.pre7_cash.mean(), prepost.post7_cash.mean())


Pre-7d payer rate: 0.012883868690434168
Post-7d payer rate: 0.013413342746205436
Payer lift (pp): 0.05294740557712681
Pre/post average cash per targeted account: 1060.624034592305 987.0389763501588


## 7. Investment hurdle

Because the historical targeting signal is observational, it is not used as a causal forecast. Instead, calculate the amount of incremental payer lift required for a ₹10 Cr annual investment to break even, plus transparent planning scenarios.


In [8]:
complete_targeting = targeting[targeting.target_date.dt.to_period('M').astype(str).le('2026-07')].copy()
monthly_targeted = complete_targeting.groupby(complete_targeting.target_date.dt.to_period('M').astype(str)).account_id.nunique()
annual_opportunities = float(monthly_targeted.mean() * 12)
july_gold = golden[(golden.payment_status.eq('SUCCESS')) & golden.audited_recovery.eq(1) & golden.event_at.dt.to_period('M').astype(str).eq('2026-07')]
july_targets = targeting[targeting.target_date.dt.to_period('M').astype(str).eq('2026-07')][['account_id']].drop_duplicates()
july_targeted_gold = july_gold.merge(july_targets, on='account_id', how='inner')
avg_recovery_per_targeted_payer_july = float(july_targeted_gold.amount.sum() / july_targeted_gold.account_id.nunique())
budget = 100_000_000
base = annual_opportunities * avg_recovery_per_targeted_payer_july
break_even_lift_pp = 100 * budget / base
print(f'Observed targeting window: 7 complete months (Jan-Jul); Aug partial excluded from annualization')
print(f'Annualized account-month opportunities: {annual_opportunities:,.0f}')
print(f'July audited recovery per targeted payer: ₹{avg_recovery_per_targeted_payer_july:,.0f}')
print(f'Break-even incremental payer lift: {break_even_lift_pp:.2f} pp')
for pp in [0.5, 1.0, break_even_lift_pp, 2.5]:
    rec = base * pp / 100
    roi = (rec-budget)/budget
    print(f'{pp:.2f} pp scenario: ₹{rec/1e7:.2f} Cr, ROI {roi:.1%}')


Observed targeting window: 7 complete months (Jan-Jul); Aug partial excluded from annualization
Annualized account-month opportunities: 67,104
July audited recovery per targeted payer: ₹77,866
Break-even incremental payer lift: 1.91 pp
0.50 pp scenario: ₹2.61 Cr, ROI -73.9%
1.00 pp scenario: ₹5.23 Cr, ROI -47.7%
1.91 pp scenario: ₹10.00 Cr, ROI 0.0%
2.50 pp scenario: ₹13.06 Cr, ROI 30.6%


## 8. Counterfactual design

The production test should randomize a 5-10% holdout within DPD x risk x loan-type strata. Treatment is assigned **before** the outcome window. Primary statistical endpoint: 30-day payer rate; financial decision metric: 30-day audited cash per eligible account. Guardrails: 60/90-day PTP kept, complaints, contact rate, and spillover. Guardrails: 60/90-day PTP kept, complaints, contact rate, and spillover. Publish confidence intervals and treatment/control balance diagnostics.


## 9. Metric governance

The requested operational metrics are now explicitly defined in `reports/Metric_Dictionary.md`. Contact rate, RPC, PTP rate, PTP kept rate, recovery/yield, recovery per targeted account, and recovery per agent-hour are calculated with fixed denominators. Cost per ₹ recovered is not fabricated because the raw package does not contain a dependable cost table.

In [9]:
from pathlib import Path
ops = pd.read_csv(REPO_ROOT / 'reports' / 'results' / 'operational_metrics_monthly.csv')
ops[['month','contact_rate','rpc_rate','ptp_rate','ptp_kept_rate','recovery_rate','recovery_per_targeted_account','recovery_per_agent_hour']].tail(7)

,month,contact_rate,rpc_rate,ptp_rate,ptp_kept_rate,recovery_rate,recovery_per_targeted_account,recovery_per_agent_hour
1,2026-02,0.194715,0.195323,0.344013,0.496786,0.009681,3925.346967,11475.961028
2,2026-03,0.196583,0.191217,0.338118,0.497202,0.012114,4925.620431,12183.453146
3,2026-04,0.198542,0.185376,0.348609,0.508656,0.010418,4133.121295,11658.080398
4,2026-05,0.194967,0.187877,0.329053,0.502658,0.010365,4178.967217,12052.895925
5,2026-06,0.194448,0.192926,0.348333,0.482619,0.010660,4300.900369,11738.743675
6,2026-07,0.200426,0.181684,0.312195,0.491311,0.011918,4782.479818,11653.783893
7,2026-08,0.190487,0.181273,0.397351,0.507756,0.002010,804.744425,12843.566362


The project makes the required statistical checks explicit in `reports/Statistical_Investigation.md`: mix, cohort, selection, survivorship/denominator, Simpson's paradox, attribution-window, and time-series effects. The available data covers 8 observed calendar months (7 complete plus partial August), so annual seasonality cannot be assessed. The experiment plan uses a 10% holdout as the preferred operating point because the 1.91 pp break-even hurdle implies roughly 3,464 total accounts under the stated planning assumptions; this is a planning calculation to be recalibrated against the actual 30-day endpoint.


**Decision:** Run a randomized targeting pilot. Do not commit ₹10 Cr until the experiment demonstrates incremental lift above the ~1.91 pp annual break-even hurdle.
